Extract N-best hypotheses from **batch transcriptions** (only returns best one).

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
AZURE_SPEECH_KEY = os.getenv("SPEECHSDK_API_KEY")
AZURE_SERVICE_REGION = os.getenv("SPEECHSDK_REGION")
AZURE_TEST_AUDIO_URL = os.getenv("AZURE_TEST_AUDIO_URL")

In [2]:
import requests
import json

# Define the endpoint URL for batch transcription
endpoint = f"https://{AZURE_SERVICE_REGION}.api.cognitive.microsoft.com/speechtotext/v3.2/transcriptions"

# Define headers
headers = {
    "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
    "Content-Type": "application/json"
}

# Define the request body
body = {
    "contentUrls": [AZURE_TEST_AUDIO_URL],  # URL to your audio file
    "locale": "en-US",  # Set the language for transcription
    "displayName": "Batch Transcription Example",
    "properties": {
        "diarizationEnabled": True,
        "wordLevelTimestampsEnabled": True,
        "punctuationMode": "automatic",
        # "profanityFilterMode": "masked",
        "outputFormat": "detailed"
    }
}

# Send the POST request
response = requests.post(endpoint, headers=headers, data=json.dumps(body))
print(response.text)  

{
  "self": "https://southeastasia.api.cognitive.microsoft.com/speechtotext/v3.2/transcriptions/813edb89-b6c7-4259-909d-623d54d41c70",
  "model": {
    "self": "https://southeastasia.api.cognitive.microsoft.com/speechtotext/v3.2/models/base/10e98dd4-3d36-4296-b383-3508d63b1e0b"
  },
  "links": {
    "files": "https://southeastasia.api.cognitive.microsoft.com/speechtotext/v3.2/transcriptions/813edb89-b6c7-4259-909d-623d54d41c70/files"
  },
  "properties": {
    "diarizationEnabled": true,
    "wordLevelTimestampsEnabled": true,
    "displayFormWordLevelTimestampsEnabled": false,
    "channels": [
      0,
      1
    ],
    "punctuationMode": "Automatic",
    "profanityFilterMode": "Masked"
  },
  "lastActionDateTime": "2024-11-13T17:43:22Z",
  "status": "NotStarted",
  "createdDateTime": "2024-11-13T17:43:22Z",
  "locale": "en-US",
  "displayName": "Batch Transcription Example"
}


In [5]:
import time

operation_url = response.headers['Location']

while True:
    status_response = requests.get(operation_url, headers=headers)
    status_data = status_response.json()

    if status_data["status"] in ["Succeeded", "Failed"]:
        break  # Exit loop if transcription is done
    print("Transcription in progress...")
    time.sleep(20)  # Poll every 20 seconds

# Print the final status
print("Transcription Status:", status_data["status"])

Transcription Status: Succeeded


In [29]:
if status_data["status"] == "Succeeded":
    result_url = status_data['links']['files']
    transcription_run_url = requests.get(result_url, headers=headers).json()['values'][0]['links']['contentUrl']
    transcription_result = requests.get(transcription_run_url, headers=headers).json()

In [31]:
transcription_result

{'source': 'https://nuscapstonewhisper.blob.core.windows.net/simulatedmedicalexams/Audio Recordings/CAR0001.mp3?sp=r&st=2024-11-13T17:42:35Z&se=2024-11-14T01:42:35Z&spr=https&sv=2022-11-02&sr=b&sig=DqRH7qs5MI5cQbfXlryXn14TLsFDFshpNZHxWp%2FljMI%3D',
 'timestamp': '2024-11-13T17:45:14Z',
 'durationInTicks': 6259000000,
 'duration': 'PT10M25.9S',
 'combinedRecognizedPhrases': [{'channel': 0,
   'lexical': "what brought you in today sure i'm just having a lot of chest pain and so i thought i should get it checked out OK and before we start could you remind me of your gender and age sure i'm thirty nine i'm a male OK and so when did this chest pain start it started last night but it's becoming sharper OK and where is this pain located it's located on the left side of my chest OK and so how long has it been going on for then if it started last night so i guess it would be a couple hours now maybe like eight OK has it been constant throughout that time or or changing i would say it's been pre

Extract N-best hypotheses from **real-time transcriptions**.

In [24]:
import json
import azure.cognitiveservices.speech as speechsdk

# Initialize speech configuration
speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)
speech_config.speech_recognition_language="en-SG"

speech_config.set_property(property_id=speechsdk.PropertyId.SpeechServiceResponse_RequestDetailedResultTrueFalse, value="true")

# Specify the path to your audio file (WAV format)
audio_file_path = "../data/primock/day1_consultation01_doctor.wav"
audio_config = speechsdk.AudioConfig(filename=audio_file_path)

# Create a recognizer with audio file input
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
result_json = []

def speech_recognizer_session_started_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStarted event')

def stop_cb(evt: speechsdk.SessionEventArgs):
    print('CLOSING on {}'.format(evt))

# Function to handle results
def handle_result(evt):
    if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print('\tText={}'.format(evt.result.text))
        result_json.append(json.loads(evt.result.json))
        if hasattr(evt, "nbest"):
            for i, hypothesis in enumerate(evt.nbest):
                print(f"Hypothesis {i+1}: {hypothesis.text} (Confidence: {hypothesis.confidence})")
        else:
            print("Only one hypothesis returned:", evt.text)

# Connect the result handler
recognizer.recognized.connect(handle_result)
recognizer.session_started.connect(speech_recognizer_session_started_cb)
recognizer.session_stopped.connect(stop_cb)

# Start recognition and wait for it to complete
recognizer.start_continuous_recognition_async()

# Allow the recognition to run in the background for a while, as it will process the whole audio file
import time
print("Recognizing...")
try:
    time.sleep(10)  # Adjust the sleep time based on the audio length
finally:
    recognizer.stop_continuous_recognition_async()

Recognizing...
SessionStarted event
	Text=Hello.
	Text=Hi. Yeah. OK. Hello. Good morning. So how can I help you this morning?


CLOSING on SessionEventArgs(session_id=09f0808c8dff44f39c20800d77c43c1e)


In [30]:
[hypothesis['Display'] for hypothesis in result_json[1]['NBest']]

['Hi. Yeah. OK. Hello. Good morning. So how can I help you this morning?',
 'hi can you stop yeah OK hello good morning sir how can i help you this morning',
 'hi can you stop yeah OK hello good morning so how can i help you this morning',
 'hi if you stop yeah OK hello good morning so how can i help you this morning',
 'hi if you stop yeah OK hello good morning sir how can i help you this morning']

Pick up multi-lingual sentence from microphone to see alternative hypotheses.

Example: 明天再回来吧, and we might need to start you on some 药物 to lower your thyroid hormones.

In [14]:
import json
import azure.cognitiveservices.speech as speechsdk

# Multilingual recognizer
speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)
speech_config.set_property(property_id=speechsdk.PropertyId.SpeechServiceResponse_RequestDetailedResultTrueFalse, value="true")
auto_detect_source_language_config = speechsdk.languageconfig.AutoDetectSourceLanguageConfig(languages=["en-US", "zh-CN"]) 
multilingual_recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config, 
    auto_detect_source_language_config=auto_detect_source_language_config
)
multilingual_result_json = []

# English (Singapore) recognizer
speech_config_en = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)
speech_config_en.speech_recognition_language = "en-SG"
speech_config_en.set_property(property_id=speechsdk.PropertyId.SpeechServiceResponse_RequestDetailedResultTrueFalse, value="true")
recognizer_en = speechsdk.SpeechRecognizer(speech_config=speech_config_en)
en_result_json = []

# Mandarin (Chinese) recognizer
speech_config_cn = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SERVICE_REGION)
speech_config_cn.speech_recognition_language = "zh-CN"
speech_config_cn.set_property(property_id=speechsdk.PropertyId.SpeechServiceResponse_RequestDetailedResultTrueFalse, value="true")
recognizer_cn = speechsdk.SpeechRecognizer(speech_config=speech_config_cn)
cn_result_json = []

def speech_recognizer_session_started_cb(evt: speechsdk.SessionEventArgs):
    print('SessionStarted event')

def stop_cb(evt: speechsdk.SessionEventArgs):
    print('CLOSING on {}'.format(evt))

# Function to handle results
def get_handler(language=""):
    def handle_result(evt):
        if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
            print('\tText={}'.format(evt.result.text))

            if language == "en":
                result_json = en_result_json
            elif language == "cn":
                result_json = cn_result_json
            else:
                result_json = multilingual_result_json

            result_json.append(json.loads(evt.result.json))
            if hasattr(evt, "nbest"):
                for i, hypothesis in enumerate(evt.nbest):
                    print(f"Hypothesis {i+1}: {hypothesis.text} (Confidence: {hypothesis.confidence})")
            else:
                print("Only one hypothesis returned:", evt.text)

    return handle_result

# Connect the result handler
multilingual_recognizer.recognized.connect(get_handler())
multilingual_recognizer.session_started.connect(speech_recognizer_session_started_cb)
multilingual_recognizer.session_stopped.connect(stop_cb)

recognizer_en.recognized.connect(get_handler(language="en"))
recognizer_en.session_started.connect(speech_recognizer_session_started_cb)
recognizer_en.session_stopped.connect(stop_cb)

recognizer_cn.recognized.connect(get_handler(language="cn"))
recognizer_cn.session_started.connect(speech_recognizer_session_started_cb)
recognizer_cn.session_stopped.connect(stop_cb)

# Start recognition and wait for it to complete
multilingual_recognizer.start_continuous_recognition_async()
recognizer_cn.start_continuous_recognition_async()
recognizer_en.start_continuous_recognition_async()

# Allow the recognition to run in the background for a while, as it will process the whole audio file
import time
print("Recognizing...")
try:
    time.sleep(15)  # Adjust the sleep time based on the audio length
finally:
    multilingual_recognizer.stop_continuous_recognition_async()
    recognizer_cn.stop_continuous_recognition_async()
    recognizer_en.stop_continuous_recognition_async()

Recognizing...
SessionStarted event
SessionStarted event
SessionStarted event
	Text=明天再回来吧。and we might need to start you on some Yahoo to lower your thyroid hormones。
	Text=明天再回来吧。and we might need to start you on some Yahoo to lower your thyroid hormones。
	Text=Maintained I realized that and we might need to start you on some ya woo to lower your thyroid hormones.


CLOSING on SessionEventArgs(session_id=e648d8b85c5e42288b47f8e4ef79f3d5)CLOSING on SessionEventArgs(session_id=05e090e308e74b41bbf90b91194028cf)

CLOSING on SessionEventArgs(session_id=c9fd0cbe0bd74ea5a302961866dc7c60)


In [18]:
multilingual_nbest = [hyp['Display'] for hyp in multilingual_result_json[0]['NBest']]
cn_nbest = [hyp['Display'] for hyp in cn_result_json[0]['NBest']]
en_nbest = [hyp['Display'] for hyp in en_result_json[0]['NBest']]

print("Multilingual:", multilingual_nbest)
print("CN:", cn_nbest)
print("EN:", en_nbest)

Multilingual: ['明天再回来吧。and we might need to start you on some Yahoo to lower your thyroid hormones。', '明天再回来吧and remind me to start you on some yahoo to lower your thyroid hormones', '明天再回来吧and reminded to start you on some yahoo to lower your thyroid hormones', '明天再回来吧and we might need to start you on some yahoo to lower your styroid hormones', '明天再回来吧and we might need to start you on some yahoo to lower your thyroid homos']
CN: ['明天再回来吧。and we might need to start you on some Yahoo to lower your thyroid hormones。', '明天再回来吧and remind me to start you on some yahoo to lower your thyroid hormones', '明天再回来吧and we might need to start you on some yahoo to lower your styroid hormones', '明天再回来吧and remind me to start you on yahoo to lower your thyroid hormones', '明天再回来吧and we might need to start you in some yahoo to lower your thyroid hormones']
EN: ['Maintained I realized that and we might need to start you on some ya woo to lower your thyroid hormones.', 'maintained i realized that and we mig